In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load your dataset
df = pd.read_csv("E:\\Jupyter\\data center project\\cleaned_facilities.csv")   # replace with your file path

In [8]:
print(df.head())
print(df.columns)
print(df.info())

   Year  Facility_ID               Facility_Name Owner_Company    City  \
0  2019  DC-D2763E00  nap de las americas madrid     Terremark  madrid   
1  2020  DC-D2763E00  nap de las americas madrid     Terremark  madrid   
2  2021  DC-D2763E00  nap de las americas madrid     Terremark  madrid   
3  2022  DC-D2763E00  nap de las americas madrid     Terremark  madrid   
4  2023  DC-D2763E00  nap de las americas madrid     Terremark  madrid   

  Country        Facility_Type  Estimated_Capacity_MW    PUE  \
0   Spain  Enterprise/Standard                   6.24  1.975   
1   Spain  Enterprise/Standard                   6.36  1.967   
2   Spain  Enterprise/Standard                   6.47  1.928   
3   Spain  Enterprise/Standard                   6.59  1.897   
4   Spain  Enterprise/Standard                   6.70  1.869   

  Cooling_System_Type  WUE_L_per_kWh  Daily_Electricity_Usage_MWh  \
0         Evaporative          1.481                       183.62   
1         Evaporative          1

<h2> Electricity And Water Usage Prediction </h2>

In [9]:
# Ratios
df["Water_per_MW"] = df["Daily_Water_Usage_Gallons"] / df["Estimated_Capacity_MW"]
df["Electricity_per_MW"] = df["Daily_Electricity_Usage_MWh"] / df["Estimated_Capacity_MW"]

# Flags
df["High_Water_Stress"] = df["Surrounding_Water_Stress_Tier"].apply(lambda x: 1 if x in ["High","Extremely High"] else 0)

# Drop IDs and names (not useful for ML)
df_ml = df.drop(["Facility_ID","Facility_Name","Owner_Company","City"], axis=1)


In [10]:
X_water = df_ml.drop("Daily_Water_Usage_Gallons", axis=1)
y_water = df_ml["Daily_Water_Usage_Gallons"]

# Train-test split
X_train_water, X_test_water, y_train_water, y_test_water = train_test_split(X_water, y_water, test_size=0.2, random_state=42)


In [11]:
categorical_features = ["Country","Facility_Type","Cooling_System_Type","Surrounding_Water_Stress_Tier"]
numeric_features = ["Year","Estimated_Capacity_MW","PUE","WUE_L_per_kWh","Daily_Electricity_Usage_MWh","Water_per_MW","Electricity_per_MW","High_Water_Stress"]

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)


In [12]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=50, max_depth=15, random_state=42),
    "Gradient Boost" : GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
}

results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("regressor", model)])
    pipeline.fit(X_train_water, y_train_water)
    y_pred_water = pipeline.predict(X_test_water)
    rmse = np.sqrt(mean_squared_error(y_test_water, y_pred_water))
    r2 = r2_score(y_test_water, y_pred_water)
    results[name] = {"RMSE": rmse, "R²": r2}

# Display results
results_df = pd.DataFrame(results).T
print(results_df)

                            RMSE        R²
Linear Regression  204362.493011  0.747015
Random Forest       12460.119988  0.999060
Gradient Boost      11443.953055  0.999207


In [13]:
X_elec = df_ml.drop("Daily_Electricity_Usage_MWh", axis=1)
y_elec = df_ml["Daily_Electricity_Usage_MWh"]

X_train_elec, X_test_elec, y_train_elec, y_test_elec = train_test_split(
    X_elec, y_elec, test_size=0.2, random_state=42
)

In [14]:
categorical_features = ["Country","Facility_Type","Cooling_System_Type","Surrounding_Water_Stress_Tier"]
numeric_features = ["Year","Estimated_Capacity_MW","PUE","WUE_L_per_kWh","Daily_Water_Usage_Gallons","Water_per_MW","Electricity_per_MW","High_Water_Stress"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=50, max_depth=15, n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
}

results_elec = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("regressor", model)])
    pipeline.fit(X_train_elec, y_train_elec)
    y_pred_elec = pipeline.predict(X_test_elec)
    rmse = np.sqrt(mean_squared_error(y_test_elec, y_pred_elec))
    r2 = r2_score(y_test_elec, y_pred_elec)
    results_elec[name] = {"RMSE": rmse, "R²": r2}

results_elec_df = pd.DataFrame(results_elec).T
print(results_elec_df)

In [ ]:
# For water prediction
numeric_features_water = [
    "Year",
    "Estimated_Capacity_MW",
    "PUE",
    "WUE_L_per_kWh",
    "Daily_Electricity_Usage_MWh",   
    "Water_per_MW",
    "Electricity_per_MW",
    "High_Water_Stress"
]

# For electricity prediction
numeric_features_elec = [
    "Year",
    "Estimated_Capacity_MW",
    "PUE",
    "WUE_L_per_kWh",
    "Daily_Water_Usage_Gallons",    
    "Water_per_MW",
    "Electricity_per_MW",
    "High_Water_Stress"
]


In [ ]:
#  Water Model Feature Importance 
rf_water = RandomForestRegressor(n_estimators=50, max_depth=15, n_jobs=-1, random_state=42)
pipeline_water = Pipeline(steps=[("preprocessor", ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("num", StandardScaler(), numeric_features_water)
])), ("regressor", rf_water)])

pipeline_water.fit(X_train, y_train)

importances_water = pipeline_water.named_steps["regressor"].feature_importances_
feature_names_water = pipeline_water.named_steps["preprocessor"].get_feature_names_out()
feat_imp_water = pd.Series(importances_water, index=feature_names_water).sort_values(ascending=False).head(15)



In [ ]:
# Electricity Model Feature Importance 
rf_elec = RandomForestRegressor(n_estimators=50, max_depth=15, n_jobs=-1, random_state=42)

pipeline_elec = Pipeline(steps=[("preprocessor", ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("num", StandardScaler(), numeric_features_elec)
])), ("regressor", rf_elec)])

pipeline_elec.fit(X_train_elec, y_train_elec)

# Extract feature importances
importances_elec = pipeline_elec.named_steps["regressor"].feature_importances_
feature_names_elec = pipeline_elec.named_steps["preprocessor"].get_feature_names_out()
feat_imp_elec = pd.Series(importances_elec, index=feature_names_elec).sort_values(ascending=False).head(15)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18,6))

# Water importance
sns.barplot(x=feat_imp_water.values, y=feat_imp_water.index, ax=axes[0])
axes[0].set_title("Top Features Driving Water Consumption")

# Electricity importance
sns.barplot(x=feat_imp_elec.values, y=feat_imp_elec.index, ax=axes[1])
axes[1].set_title("Top Features Driving Electricity Consumption")

plt.tight_layout()
plt.show()


<h2> Risk Classifier

In [ ]:
# Define threshold for water intensity (example: > 1000 gallons per MW per day)
threshold = 1000

df_ml["Risk_Label"] = np.where(
    (df_ml["High_Water_Stress"] == 1) & (df_ml["Water_per_MW"] > threshold),
    1,  # High risk
    0   # Low risk
)


In [ ]:
X = df_ml.drop("Risk_Label", axis=1)
y = df_ml["Risk_Label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
categorical_features = ["Country","Facility_Type","Cooling_System_Type","Surrounding_Water_Stress_Tier"]
numeric_features = ["Year","Estimated_Capacity_MW","PUE","WUE_L_per_kWh","Daily_Electricity_Usage_MWh","Daily_Water_Usage_Gallons","Water_per_MW","Electricity_per_MW","High_Water_Stress"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
}

results = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    }

results_df = pd.DataFrame(results).T
print(results_df)
